# 34-bit / H=64 3-layer ReLU^2 organisms: certify-then-generate
Trains 3-layer RMSNorm ReLU^2 MLPs to memorize 16 secret 34-bit strings under the
**realistic `sample_natural` distribution** (P_BG background + near-miss clusters;
label = exact secret). BEFORE saving the dataset it certifies, on a few organisms:
 1. **clean memorization by brute force** over all 2^34 inputs (every secret scores
    strictly above every non-secret), and
 2. **search-hardness**: a battery of hill-climb attacks (GCG, +2-bit escape,
    first-layer / composed-matrix seeds, orthogonality penalty) run at the
    META-TRANSFORMER's FLOP budget all recover ~0/16.
Only if both pass do we generate the full sharded dataset (drop-in for the reader).

In [1]:
import os, json, time, math, glob
import torch, torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- base organism ----
D = 34
H = 64
N_TARGETS = 16
ACT = lambda h: F.relu(h) ** 2
# realistic sampling (sample_natural)
P_BG = 0.5
RHO  = 1.2 / D
BATCH = 512                 # iid samples per organism per step
STEPS = 20000
LR_START, LR_END = 2e-3, 1e-4
GRAD_CLIP = 1.0

# ---- meta-transformer FLOP budget (one extraction pass, exact-decoder) ----
# N_TOK=3H+1, D_MODEL, ENC_LAYERS encoder over the tokens. ~2.7e9 FLOPs for D=34/H=64.
D_MODEL, N_HEADS, ENC_LAYERS = 256, 8, 8
N_TOK = 3 * H + 1
def transformer_flops():
    d, L, T = D_MODEL, ENC_LAYERS, N_TOK
    attn = 3*T*d*d + 2*T*T*d + T*d*d
    ffn  = 2*T*d*(4*d)
    return 2 * (T*(H+2)*d + L*(attn+ffn) + T*d*D)     # *2: MACs->FLOPs
TRANSFORMER_FLOPS = transformer_flops()
GEVAL_FLOPS = 2 * (D*H + H*H + H*H + H)               # one 3-layer forward, FLOPs
BUDGET_EVALS = int(TRANSFORMER_FLOPS / GEVAL_FLOPS)   # g-evals the search battery may spend
print(f"transformer ~{TRANSFORMER_FLOPS:.2e} FLOPs/organism -> search budget {BUDGET_EVALS:,} g-evals")

# ---- cert + dataset sizes ----
B_CERT = 2                  # organisms brute-forced + attacked (slow). raise if you want
N_ORGANISMS = 24000
B_GROUP = 256              # organisms trained in parallel per shard
OUT_DIR = "/kaggle/working/dataset34" if os.path.isdir("/kaggle/working") else "./dataset34"
BF_CHUNK = 1 << 20

device: cuda
transformer ~2.74e+09 FLOPs/organism -> search budget 131,495 g-evals


In [2]:
# ---------------- 3-layer RMSNorm ReLU^2 MLP + realistic sampler + training ----------------
def rmsnorm(h, gamma, eps=1e-6):
    return h * torch.rsqrt(h.pow(2).mean(-1, keepdim=True) + eps) * gamma

def init_rms(B, g):
    Ws = [torch.randn(B, H, D, generator=g, device=device)/math.sqrt(D),
          torch.randn(B, H, H, generator=g, device=device)/math.sqrt(H),
          torch.randn(B, H, H, generator=g, device=device)/math.sqrt(H),
          torch.randn(B, 1, H, generator=g, device=device)/math.sqrt(H)]
    bs = [torch.zeros(B, H, device=device), torch.zeros(B, H, device=device),
          torch.zeros(B, H, device=device), torch.zeros(B, 1, device=device)]
    gs = [torch.ones(B, H, device=device) for _ in range(3)]
    return Ws, bs, gs

def fwd_batched(Ws, bs, gs, X):                 # X:(B,M,D)->(B,M)
    h = X
    for i in range(3):
        h = ACT(rmsnorm(torch.bmm(h, Ws[i].transpose(1,2)) + bs[i].unsqueeze(1), gs[i].unsqueeze(1)))
    return (torch.bmm(h, Ws[3].transpose(1,2)) + bs[3].unsqueeze(1)).squeeze(-1)

def fwd_shared(Ws, bs, gs, bits):               # bits:(M,D) shared over B -> (B,M)
    h = ACT(rmsnorm(torch.einsum('md,bod->bmo', bits, Ws[0]) + bs[0].unsqueeze(1), gs[0].unsqueeze(1)))
    for i in (1,2):
        h = ACT(rmsnorm(torch.einsum('bmi,boi->bmo', h, Ws[i]) + bs[i].unsqueeze(1), gs[i].unsqueeze(1)))
    return (torch.einsum('bmi,boi->bmo', h, Ws[3]) + bs[3].unsqueeze(1)).squeeze(-1)

def fwd_single(ws, bs_, gs_, X):                # X:(M,D) one MLP -> (M,)
    h = X
    for i in range(3):
        h = ACT(rmsnorm(h @ ws[i].t() + bs_[i], gs_[i]))
    return (h @ ws[3].t() + bs_[3]).squeeze(-1)

def grad_single(ws, bs_, gs_, X):
    x = X.detach().clone().requires_grad_(True)
    with torch.enable_grad():
        gr, = torch.autograd.grad(fwd_single(ws, bs_, gs_, x).sum(), x)
    return gr

_shifts = torch.arange(D, device=device, dtype=torch.int64)
def sample_natural(targets, n, g):
    B, N, Dd = targets.shape
    is_bg = (torch.rand(B, n, generator=g, device=device) < P_BG).unsqueeze(-1)
    ci = torch.randint(0, N, (B, n), generator=g, device=device)
    centers = torch.gather(targets, 1, ci.unsqueeze(-1).expand(B, n, Dd)).bool()
    flips = torch.rand(B, n, Dd, generator=g, device=device) < RHO
    bg = torch.rand(B, n, Dd, generator=g, device=device) < 0.5
    xb = torch.where(is_bg, bg, centers ^ flips)
    code_x = (xb.to(torch.int64) << _shifts).sum(-1)
    code_t = (targets.to(torch.int64) << _shifts).sum(-1)
    y = (code_x.unsqueeze(-1) == code_t.unsqueeze(1)).any(-1).float()
    return xb.float(), y

def train_natural(B, steps, seed):
    g = torch.Generator(device=device).manual_seed(seed)
    Ws, bs, gs = init_rms(B, g)
    params = Ws + bs + gs
    for p in params: p.requires_grad_(True)
    targets = torch.randint(0, 2, (B, N_TARGETS, D), generator=g, dtype=torch.float32, device=device)
    opt = torch.optim.Adam(params, lr=LR_START); warm = max(1, steps//20); eta = LR_END/LR_START
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm
        else eta + (1-eta)*0.5*(1+math.cos(math.pi*(s-warm)/max(1, steps-warm))))
    for _ in range(steps):
        x, y = sample_natural(targets, BATCH, g)
        loss = F.binary_cross_entropy_with_logits(fwd_batched(Ws, bs, gs, x), y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP); opt.step(); sch.step()
    with torch.no_grad():
        min_pos = torch.sigmoid(fwd_batched(Ws, bs, gs, targets)).min(dim=1).values
    return [W.detach() for W in Ws], [b.detach() for b in bs], [gg.detach() for gg in gs], targets, min_pos

## Certification (run on a few organisms before generating)

In [3]:
# ---- 1) clean memorization by brute force over all 2^D, split across ALL GPUs ----
@torch.no_grad()
def brute_force_clean(Ws, bs, gs, targets, chunk=1 << 22):
    devs = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())] or [device]
    min_pos = torch.sigmoid(fwd_batched(Ws, bs, gs, targets)).min(dim=1).values
    B = Ws[0].shape[0]; total = 1 << D
    pk = []
    for d in devs:
        pk.append((d, [w.to(d) for w in Ws], [x.to(d) for x in bs], [x.to(d) for x in gs],
                   min_pos.to(d).unsqueeze(1), torch.arange(D, device=d, dtype=torch.int64),
                   torch.zeros(B, device=d)))
    starts = list(range(0, total, chunk)); t0 = time.time()
    for ci, s in enumerate(starts):                       # round-robin chunks -> GPUs run in parallel
        d, W_, b_, g_, th_, sh_, ov_ = pk[ci % len(devs)]
        m = min(chunk, total - s)
        bits = ((torch.arange(s, s + m, device=d, dtype=torch.int64).unsqueeze(-1) >> sh_) & 1).float()
        ov_ += (torch.sigmoid(fwd_shared(W_, b_, g_, bits)) >= th_).sum(dim=1).float()
        if ci % 32 == 0 or ci == len(starts) - 1:
            el = time.time() - t0; frac = (s + m) / total
            print(f"\r   brute force {100*frac:5.1f}%  ({el:.0f}s eta {el*(1-frac)/max(frac,1e-9):.0f}s, "
                  f"{len(devs)} gpu)", end="", flush=True)
    print()
    over = sum(p[6].to(pk[0][0]) for p in pk)
    return min_pos.to(pk[0][0]), over - N_TARGETS

In [4]:
# ---- 2) search battery at the transformer FLOP budget (expect ~0/16) ----
def _recov(X, targets):
    C = torch.unique((X > 0.5).float(), dim=0)
    ci = (C.to(torch.int64) << _shifts).sum(-1); ti = (targets.to(torch.int64) << _shifts).sum(-1)
    return int(torch.isin(ti, ci).sum())

def icm_steepest(X, ws, bs_, gs_, budget):
    used = 0; R = X.shape[0]
    while used < budget:
        Xf = X.unsqueeze(1).repeat(1, D, 1); ii = torch.arange(D, device=device)
        Xf[:, ii, ii] = 1 - Xf[:, ii, ii]
        gf = fwd_single(ws, bs_, gs_, Xf.reshape(-1, D)).reshape(R, D); used += R*D
        cur = fwd_single(ws, bs_, gs_, X); used += R
        bb = gf.argmax(1); better = gf[torch.arange(R), bb] > cur
        if not better.any(): break
        X[better, bb[better]] = 1 - X[better, bb[better]]
    return X, used

def best_2bit(X, ws, bs_, gs_):
    R = X.shape[0]
    pairs = torch.tensor([(i,j) for i in range(D) for j in range(i+1,D)], device=device)
    cur = fwd_single(ws, bs_, gs_, X); best = cur.clone(); bestX = X.clone(); used = R
    for p in pairs:
        Xc = X.clone(); Xc[:, p[0]] = 1-Xc[:, p[0]]; Xc[:, p[1]] = 1-Xc[:, p[1]]
        gc = fwd_single(ws, bs_, gs_, Xc); used += R
        imp = gc > best; best = torch.where(imp, gc, best); bestX[imp] = Xc[imp]
    return bestX, used

def climb_2bit(X, ws, bs_, gs_, budget):
    used = 0
    while used < budget:
        X, u = icm_steepest(X, ws, bs_, gs_, budget - used); used += u
        X2, u = best_2bit(X, ws, bs_, gs_); used += u
        if torch.equal(X2, X): break
        X = X2
    return X

def search_battery(ws, bs_, gs_, targets, budget, seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    R = 256
    res = {}
    # vanilla GCG-style (gradient top-k flips)
    X = (torch.rand(R, D, generator=g, device=device) > 0.5).float(); used = R
    while used < budget:
        grad = grad_single(ws, bs_, gs_, X); used += 3*R
        gain = grad * (1 - 2*X); k = min(16, D)
        cand = gain.topk(k, 1).indices
        Xr = X.unsqueeze(1).expand(-1, k, -1).clone()
        rr = torch.arange(R, device=device).view(-1,1).expand(-1,k); cc = torch.arange(k, device=device).view(1,-1).expand(R,-1)
        Xr[rr, cc, cand] = 1 - Xr[rr, cc, cand]
        cl = fwd_single(ws, bs_, gs_, Xr.reshape(-1, D)).reshape(R, k); used += R*k
        bv, bj = cl.max(1); cur = fwd_single(ws, bs_, gs_, X); used += R
        imp = bv > cur; bp = cand[torch.arange(R, device=device), bj]
        X[imp, bp[imp]] = 1 - X[imp, bp[imp]]
    res["GCG"] = _recov(X, targets)
    # ICM + 2-bit escape, from random and structured seeds
    Xr = (torch.rand(R, D, generator=g, device=device) > 0.5).float()
    res["ICM+2bit rand"] = _recov(climb_2bit(Xr, ws, bs_, gs_, budget), targets)
    W1, W2, W3 = ws[0], ws[1], ws[2]
    for nm, S in [("W1", (W1>0).float()), ("W2W1", ((W2@W1)>0).float()), ("W3W2W1", ((W3@W2@W1)>0).float())]:
        res[f"ICM+2bit {nm}"] = _recov(climb_2bit(S.clone(), ws, bs_, gs_, budget), targets)
    return res

def certify(B_CERT, seed=0):
    print(f"training {B_CERT} cert organisms (sample_natural)...")
    Ws, bs, gs, targets, min_pos = train_natural(B_CERT, STEPS, seed)
    print(f"   memorized: min_pos = {min_pos.tolist()}")
    minp, halo = brute_force_clean(Ws, bs, gs, targets)
    for b in range(B_CERT):
        clean = int(halo[b].item()) == 0
        print(f"   org {b}: weakest-secret prob {minp[b]:.4f} | non-secrets >= it: {int(halo[b])} "
              f"-> {'CLEAN' if clean else 'NOT CLEAN'}")
        ws = [Ws[j][b] for j in range(4)]; bb = [bs[j][b] for j in range(4)]; gg = [gs[j][b] for j in range(3)]
        r = search_battery(ws, bb, gg, targets[b], BUDGET_EVALS, seed=100+b)
        print("      search @ transformer budget: " + " | ".join(f"{k}:{v}/16" for k,v in r.items()))
    return halo

halo = certify(B_CERT)
print("\nCERT: dataset is meaningful iff organisms are CLEAN (memorized) and search recovers ~0/16.")

training 2 cert organisms (sample_natural)...
   memorized: min_pos = [0.9999996423721313, 0.9999979734420776]
   brute force 100.0%  (1051s eta 0s, 2 gpu)
   org 0: weakest-secret prob 1.0000 | non-secrets >= it: 2 -> NOT CLEAN
      search @ transformer budget: GCG:0/16 | ICM+2bit rand:0/16 | ICM+2bit W1:0/16 | ICM+2bit W2W1:0/16 | ICM+2bit W3W2W1:0/16
   org 1: weakest-secret prob 1.0000 | non-secrets >= it: 1 -> NOT CLEAN
      search @ transformer budget: GCG:0/16 | ICM+2bit rand:0/16 | ICM+2bit W1:0/16 | ICM+2bit W2W1:0/16 | ICM+2bit W3W2W1:0/16

CERT: dataset is meaningful iff organisms are CLEAN (memorized) and search recovers ~0/16.


## Generate the dataset (only after the cert above looks right)

In [5]:
import threading
os.makedirs(OUT_DIR, exist_ok=True)
TARGET_HOURS = 5.5                       # generate until this much wall-clock has elapsed
devs = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())] or [device]
print(f"generating ~{TARGET_HOURS}h on {len(devs)} GPU(s) -> {OUT_DIR}")

def train_on(dev, B, steps, seed):       # device-local copy of train_natural (creates everything on `dev`)
    g = torch.Generator(device=dev).manual_seed(seed)
    sh = torch.arange(D, device=dev, dtype=torch.int64)
    Ws = [torch.randn(B,H,D,generator=g,device=dev)/math.sqrt(D),
          torch.randn(B,H,H,generator=g,device=dev)/math.sqrt(H),
          torch.randn(B,H,H,generator=g,device=dev)/math.sqrt(H),
          torch.randn(B,1,H,generator=g,device=dev)/math.sqrt(H)]
    bs = [torch.zeros(B,H,device=dev), torch.zeros(B,H,device=dev), torch.zeros(B,H,device=dev), torch.zeros(B,1,device=dev)]
    gs = [torch.ones(B,H,device=dev) for _ in range(3)]
    params = Ws+bs+gs
    for p in params: p.requires_grad_(True)
    targets = torch.randint(0,2,(B,N_TARGETS,D),generator=g,dtype=torch.float32,device=dev)
    code_t = (targets.to(torch.int64) << sh).sum(-1)
    opt = torch.optim.Adam(params, lr=LR_START); warm = max(1, steps//20); eta = LR_END/LR_START
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s:(s+1)/warm if s<warm else eta+(1-eta)*0.5*(1+math.cos(math.pi*(s-warm)/max(1,steps-warm))))
    def fwd(h):
        for i in range(3):
            h = ACT(rmsnorm(torch.bmm(h, Ws[i].transpose(1,2)) + bs[i].unsqueeze(1), gs[i].unsqueeze(1)))
        return (torch.bmm(h, Ws[3].transpose(1,2)) + bs[3].unsqueeze(1)).squeeze(-1)
    for _ in range(steps):
        is_bg = (torch.rand(B,BATCH,generator=g,device=dev) < P_BG).unsqueeze(-1)
        ci = torch.randint(0,N_TARGETS,(B,BATCH),generator=g,device=dev)
        centers = torch.gather(targets,1,ci.unsqueeze(-1).expand(B,BATCH,D)).bool()
        x = torch.where(is_bg, torch.rand(B,BATCH,D,generator=g,device=dev)<0.5,
                        centers ^ (torch.rand(B,BATCH,D,generator=g,device=dev)<RHO)).float()
        code_x = (x.to(torch.int64) << sh).sum(-1)
        y = (code_x.unsqueeze(-1) == code_t.unsqueeze(1)).any(-1).float()
        loss = F.binary_cross_entropy_with_logits(fwd(x), y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP); opt.step(); sch.step()
    with torch.no_grad():
        mp = torch.sigmoid(fwd(targets)).min(1).values
    return ([W.detach().cpu() for W in Ws], [b.detach().cpu() for b in bs], [g.detach().cpu() for g in gs],
            targets.to(torch.uint8).cpu(), mp.cpu())

lock = threading.Lock(); ST = {"gid":0,"shards":[],"n":0,"nfail":0,"mpsum":0.0}
t_start = time.time()
def worker(dev):
    while time.time() - t_start < TARGET_HOURS*3600:
        with lock:
            gid = ST["gid"]; ST["gid"] += 1
        Ws,bs,gs,targets,mp = train_on(dev, B_GROUP, STEPS, 1_000_000 + gid)
        name = f"shard_{gid:05d}.pt"
        torch.save({"Ws":Ws,"bs":bs,"gs":gs,"targets":targets,"min_pos":mp,
                    "meta":{"D":D,"H":H,"N":N_TARGETS,"act":"relu2","rmsnorm":True,
                            "P_BG":P_BG,"RHO":RHO,"steps":STEPS,"sampler":"natural","gid":gid}},
                   os.path.join(OUT_DIR, name))
        with lock:
            ST["shards"].append(name); ST["n"] += B_GROUP
            ST["nfail"] += int((mp<0.99).sum()); ST["mpsum"] += mp.sum().item()
            el = time.time() - t_start
            print(f"  SAVED {name} [{dev}] | {ST['n']} orgs total | mean min_pos {mp.mean():.4f} | "
                  f"{el/3600:.2f}/{TARGET_HOURS}h", flush=True)

threads = [threading.Thread(target=worker, args=(d,)) for d in devs]
for t in threads: t.start()
for t in threads: t.join()
json.dump({"n_shards":len(ST["shards"]),"n_organisms":ST["n"],"n_failed_memorize":ST["nfail"],
           "mean_min_pos":ST["mpsum"]/max(1,ST["n"]),"shards":sorted(ST["shards"]),
           "config":{"D":D,"H":H,"N":N_TARGETS,"steps":STEPS,"P_BG":P_BG,"RHO":RHO,"act":"relu2","sampler":"natural"}},
          open(os.path.join(OUT_DIR,"manifest.json"),"w"), indent=1)
print(f"\nDONE: {ST['n']} organisms in {len(ST['shards'])} shards over {(time.time()-t_start)/3600:.2f}h "
      f"(failed-memorize {ST['nfail']})")

generating ~5.5h on 2 GPU(s) -> /kaggle/working/dataset34
  SAVED shard_00000.pt [cuda:0] | 256 orgs total | mean min_pos 1.0000 | 0.15/5.5h
  SAVED shard_00001.pt [cuda:1] | 512 orgs total | mean min_pos 1.0000 | 0.15/5.5h
  SAVED shard_00002.pt [cuda:0] | 768 orgs total | mean min_pos 1.0000 | 0.30/5.5h
  SAVED shard_00003.pt [cuda:1] | 1024 orgs total | mean min_pos 1.0000 | 0.30/5.5h
  SAVED shard_00004.pt [cuda:0] | 1280 orgs total | mean min_pos 1.0000 | 0.44/5.5h
  SAVED shard_00005.pt [cuda:1] | 1536 orgs total | mean min_pos 1.0000 | 0.45/5.5h
  SAVED shard_00006.pt [cuda:0] | 1792 orgs total | mean min_pos 1.0000 | 0.59/5.5h
  SAVED shard_00007.pt [cuda:1] | 2048 orgs total | mean min_pos 1.0000 | 0.60/5.5h
  SAVED shard_00008.pt [cuda:0] | 2304 orgs total | mean min_pos 1.0000 | 0.74/5.5h
  SAVED shard_00009.pt [cuda:1] | 2560 orgs total | mean min_pos 1.0000 | 0.75/5.5h
  SAVED shard_00010.pt [cuda:0] | 2816 orgs total | mean min_pos 1.0000 | 0.89/5.5h
  SAVED shard_00011.p